In [1]:
!pip install tiktoken langchain_google_genai python-dotenv boto3 pydantic langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.1 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import time
import boto3
import tiktoken
from collections import deque
from datetime import datetime
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

In [3]:
# ==========================================
# 1. CONFIGURACIÓN Y VARIABLES DE ENTORNO
# ==========================================
load_dotenv()
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if GEMINI_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME")
S3_PREFIX_BRONCE = os.getenv("S3_PREFIX_BRONCE")
S3_PREFIX_SILVER = os.getenv("S3_PREFIX_SILVER")
S3_PREFIX_QUARANTINE = os.getenv("S3_PREFIX_QUARANTINE")

In [4]:
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION
)

In [5]:
# ==========================================
# 2. CONTROLADOR DE PRESUPUESTO API
# ==========================================
class ControladorPresupuesto:
    def __init__(self, archivo_estado="estado_api.json", max_peticiones=20, max_tokens=250000, max_rpm=4):
        self.archivo_estado = archivo_estado
        self.max_peticiones = max_peticiones
        self.max_tokens = max_tokens
        self.max_rpm = max_rpm

        # Usamos deque para mantener un registro eficiente de los últimos timestamps
        self.historial_rpm = deque(maxlen=max_rpm)

        self.codificador = tiktoken.get_encoding("cl100k_base")
        self.estado = self._cargar_estado()

    def _cargar_estado(self):
        hoy = datetime.now().strftime("%Y-%m-%d")
        if os.path.exists(self.archivo_estado):
            with open(self.archivo_estado, 'r') as f:
                estado = json.load(f)
                if estado.get("fecha") == hoy:
                    return estado
        return {"fecha": hoy, "peticiones": 0, "tokens_usados": 0}

    def _guardar_estado(self):
        with open(self.archivo_estado, 'w') as f:
            json.dump(self.estado, f)

    def _controlar_rpm(self):
        """Verifica la ventana de 60 segundos y pausa la ejecución si es necesario."""
        ahora = time.time()

        # Limpiar del historial las peticiones que tienen más de 60 segundos
        while self.historial_rpm and ahora - self.historial_rpm[0] >= 60:
            self.historial_rpm.popleft()

        # Si ya alcanzamos el límite en el último minuto, calculamos la espera
        if len(self.historial_rpm) >= self.max_rpm:
            # El tiempo que falta para que la petición más antigua cumpla 60 segundos
            tiempo_espera = 60 - (ahora - self.historial_rpm[0])
            if tiempo_espera > 0:
                print(f"   Control RPM: Límite de {self.max_rpm}/min alcanzado. Pausando {tiempo_espera:.1f} seg...")
                time.sleep(tiempo_espera)

        # Registrar el timestamp de la petición actual que está a punto de salir
        self.historial_rpm.append(time.time())

    def autorizar_peticion(self, texto_prompt):
        tokens_estimados = len(self.codificador.encode(texto_prompt))
        tokens_totales_estimados = tokens_estimados + 500 # Margen de respuesta

        # 1. Validaciones Diarias (Hard Limits)
        if self.estado["peticiones"] >= self.max_peticiones:
            print("   ALERTA: Límite de 20 peticiones diarias alcanzado.")
            return False, 0

        if self.estado["tokens_usados"] + tokens_totales_estimados > self.max_tokens:
            print(f"  ALERTA: Exceso de límite de 250k tokens diarios.")
            return False, 0

        # 2. Validación por Minuto (Soft Limit con espera dinámica)
        self._controlar_rpm()

        return True, tokens_totales_estimados

    def registrar_consumo(self, tokens_reales_usados):
        self.estado["peticiones"] += 1
        self.estado["tokens_usados"] += tokens_reales_usados
        self._guardar_estado()
        print(f"   Consumo API: {self.estado['peticiones']}/20 req | {self.estado['tokens_usados']}/250k tokens")

controlador = ControladorPresupuesto(max_rpm=4)

In [11]:
# ==========================================
# 3. CONFIGURACIÓN LLM (LANGCHAIN + PYDANTIC)
# ==========================================
class SeccionFDS(BaseModel):
    encontrada: bool = Field(description="True si lograste identificar el inicio y fin de la Sección 4. False si no existe.")
    contenido: str = Field(description="El texto exacto de la Sección 4, desde su título hasta justo antes de que empiece la Sección 5.")

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)
llm_estructurado = llm.with_structured_output(SeccionFDS)

prompt_extractor = PromptTemplate.from_template(
    """Tu única tarea es extraer un bloque de texto específico del documento Markdown proporcionado.

    INSTRUCCIONES ESTRICTAS:
    1. Localiza dónde comienza la "Sección 4" (puede llamarse "Medidas de primeros auxilios", "Primeros auxilios", etc.).
    2. Localiza dónde comienza la "Sección 5" (puede llamarse "Medidas de lucha contra incendios", "Lucha contra incendios", etc.).
    3. Extrae y devuelve ÚNICAMENTE el texto exacto que está entre el inicio de la Sección 4 y el inicio de la Sección 5.
    4. NO modifiques el texto, NO lo resumas y NO agregues ningún comentario tuyo.

    TEXTO DEL DOCUMENTO:
    {texto_documento}
    """
)
cadena_extraccion = prompt_extractor | llm_estructurado

In [7]:
# ==========================================
# 4. FUNCIONES DE PROCESAMIENTO Y LIMPIEZA
# ==========================================
def limpiar_texto_md(texto):
    """Limpia el ruido generado por el OCR y la paginación."""
    texto = re.sub(r'!\[.*?\]\(.*?\)', '', texto) # Elimina etiquetas de imágenes residuales
    texto = re.sub(r'(?i)\*?Página \d+ de \d+\*?', '', texto)
    texto = re.sub(r'^\d+$\n', '', texto, flags=re.MULTILINE)
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()

def extraer_seccion_4_regex(texto_md):
    """Tier 1: Intento de extracción rápida sin gastar API."""
    # Busca "Sección 4" o "Medidas de primeros auxilios"
    patron_inicio = re.compile(r"(?im)^#*\s*(?:SECCI[OÓ]N\s*)?4[\.\-\:]?\s*(?:MEDIDAS\s*DE\s*)?(?:PRIMEROS\s*AUXILIOS).*$")
    # Se detiene en "Sección 5" o "Medidas de lucha contra incendios"
    patron_fin = re.compile(r"(?im)^#*\s*(?:SECCI[OÓ]N\s*)?5[\.\-\:]?\s*(?:MEDIDAS\s*DE\s*)?(?:LUCHA\s*CONTRA\s*INCENDIOS).*$")

    match_inicio = patron_inicio.search(texto_md)
    if not match_inicio: return None

    match_fin = patron_fin.search(texto_md, match_inicio.end())
    inicio_idx = match_inicio.start()
    fin_idx = match_fin.start() if match_fin else len(texto_md)

    return texto_md[inicio_idx:fin_idx].strip()

def extraer_seccion_4_llm_seguro(texto_md):
    """Tier 2: Extracción semántica protegida por cuotas."""
    print(" Regex falló. Evaluando Extracción Semántica (LLM)...")

    # Acotar a max 4000 tokens para no quemar la cuota
    tokens_doc = controlador.codificador.encode(texto_md)
    texto_recortado = controlador.codificador.decode(tokens_doc[:4000]) if len(tokens_doc) > 4000 else texto_md

    prompt_completo = prompt_extractor.format(texto_documento=texto_recortado)
    autorizado, tokens_estimados = controlador.autorizar_peticion(prompt_completo)

    if not autorizado:
        print("   Abortando LLM para este archivo por límite de cuota.")
        return None

    try:
        resultado = cadena_extraccion.invoke({"texto_documento": texto_recortado})
        controlador.registrar_consumo(tokens_estimados)
        return resultado.contenido if resultado.encontrada else None
    except Exception as e:
        print(f"   Error en la API: {e}")
        return None

In [8]:
def obtener_archivos_s3(prefijo):
    archivos = []
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=S3_BUCKET_NAME, Prefix=prefijo):
        if 'Contents' in page:
            for obj in page['Contents']:
                if not obj['Key'].endswith('/'): archivos.append(obj['Key'])
    return archivos

In [9]:
# ==========================================
# 5. ORQUESTADOR PRINCIPAL (EL PIPELINE)
# ==========================================
def procesar_pipeline_maestro():
    print(" Iniciando Extracción Híbrida: Bronce -> Silver (Sección 4)")

    # 1. Obtener listas de control
    archivos_silver = obtener_archivos_s3(f"{S3_PREFIX_SILVER}seccion_4/")
    archivos_quarantine = obtener_archivos_s3(S3_PREFIX_QUARANTINE)

    ids_procesados = [os.path.basename(key).replace('.jsonl', '') for key in archivos_silver]
    ids_cuarentena = [os.path.basename(key).replace('.md', '') for key in archivos_quarantine]

    # 2. Obtener TODO lo que hay en Bronce y filtrar SOLO los archivos .md
    todos_archivos_bronce = obtener_archivos_s3(S3_PREFIX_BRONCE)
    mds_en_bronce = [key for key in todos_archivos_bronce if key.endswith('.md')]

    for s3_key in mds_en_bronce:
        nombre_md = os.path.basename(s3_key)
        doc_id = nombre_md.replace('.md', '')

        if doc_id in ids_procesados:
            print(f" Saltando: {doc_id} (Ya existe en Silver/seccion_4)")
            continue
        if doc_id in ids_cuarentena:
            print(f" Saltando: {doc_id} (Marcado en Cuarentena previamente)")
            continue

        print(f"\n Analizando: {nombre_md}")
        response = s3_client.get_object(Bucket=S3_BUCKET_NAME, Key=s3_key)
        texto_md = response['Body'].read().decode('utf-8')

        # 1. Intentar Regex (Tier 1)
        contenido = extraer_seccion_4_regex(texto_md)

        # 2. Si falla Regex, intentar LLM (Tier 2)
        if not contenido:
            contenido = extraer_seccion_4_llm_seguro(texto_md)

        # 3. Guardar en Silver o mandar a Quarantine
        if contenido:
            contenido_limpio = limpiar_texto_md(contenido)
            print(f" Sección extraída y limpia. Guardando en Silver...")

            # Insertamos explícitamente "seccion": 4
            datos = {"doc_id": doc_id, "seccion": 4, "contenido": contenido_limpio}
            local_path = f"/content/{doc_id}.jsonl"
            with open(local_path, 'w', encoding='utf-8') as f:
                f.write(json.dumps(datos, ensure_ascii=False) + '\n')

            # Guardamos en la ruta correcta de S3
            s3_client.upload_file(local_path, S3_BUCKET_NAME, f"{S3_PREFIX_SILVER}seccion_4/{doc_id}.jsonl")
            os.remove(local_path)
        else:
            print(f"   Formato irreconocible o sin saldo. Moviendo a Quarantine...")
            s3_client.copy_object(
                Bucket=S3_BUCKET_NAME,
                CopySource={'Bucket': S3_BUCKET_NAME, 'Key': s3_key},
                Key=f"{S3_PREFIX_QUARANTINE}{nombre_md}"
            )

In [13]:
procesar_pipeline_maestro()

 Iniciando Extracción Híbrida: Bronce -> Silver (Sección 4)

 Analizando: Esmalte Uretano AR Comp. B.md
 Sección extraída y limpia. Guardando en Silver...

 Analizando: FDS 1 -pintura-antideslizante-para-canchas.md
 Sección extraída y limpia. Guardando en Silver...

 Analizando: FDS 10 - pintura-epoxi-poliamida-blanco-13243-10012925-10344427-hoja-de-seguridad-pdf.md
 Sección extraída y limpia. Guardando en Silver...

 Analizando: FDS 11  - Ecosphere Premium.md
 Sección extraída y limpia. Guardando en Silver...

 Analizando: FDS 12 - PINTURA ANTICORROSIVA GRIS 507 _ 10014333-10012454-10171712 _COL (Versión 3) (1).md
 Sección extraída y limpia. Guardando en Silver...

 Analizando: FDS 13 -AVANTEX-FACHADA.md
 Sección extraída y limpia. Guardando en Silver...

 Analizando: FDS 14 -PINTURA-ACRILICA-PARA-TRÁFICO.md
 Sección extraída y limpia. Guardando en Silver...

 Analizando: FDS 15 - impra.md
 Sección extraída y limpia. Guardando en Silver...

 Analizando: FDS 16 - sellador-nitrocelulosi